In [1]:
#Importación de librerías
import pyspark
from pyspark.sql import SparkSession, Row

from pyspark.sql.types import MapType, StringType
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

In [2]:
#Creación de sesión
spark = SparkSession \
    .builder \
    .appName("HousingAnalysis") \
    .getOrCreate()

In [3]:
#Carga de datos
path_archivo = '/Users/Nefi/OneDrive/Documentos/09-EBAC/EBACMX-DATA-ANALYST/Referencias/kc_house_data_clean.csv'
df = spark.read.csv(path_archivo, header=True, inferSchema=True)

In [4]:
df

DataFrame[id: bigint, price: int, bedrooms: int, bathrooms: double, sqft_living: int, sqft_lot: int, floors: double, waterfront: int, view: int, condition: int, grade: int, sqft_above: int, sqft_basement: int, yr_built: int, yr_renovated: int, zipcode: int, lat: double, long: double, sqft_living15: int, sqft_lot15: int, date: date, sale_year: int, price_per_m2: double]

In [5]:
df.show()

+----------+-------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+----------+---------+------------+
|        id|  price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|      date|sale_year|price_per_m2|
+----------+-------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+----------+---------+------------+
|7129300520| 221900|       3|      1.0|       1180|    5650|   1.0|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|2014-10-13|     2014|     2024.16|
|6414100192| 538000|       3|     2.25|       2570|    7242|   2.0|         

In [6]:
df.count()

21613

In [7]:
#Primera visualización
df.createOrReplaceTempView("housing")

In [8]:
sql_str = "select zipcode, count(*) as total_casas from housing group by zipcode order by total_casas desc"
spark.sql(sql_str).show()

+-------+-----------+
|zipcode|total_casas|
+-------+-----------+
|  98103|        602|
|  98038|        590|
|  98115|        583|
|  98052|        574|
|  98117|        553|
|  98042|        548|
|  98034|        545|
|  98118|        508|
|  98023|        499|
|  98006|        498|
|  98133|        494|
|  98059|        468|
|  98058|        455|
|  98155|        446|
|  98074|        441|
|  98033|        432|
|  98027|        412|
|  98125|        410|
|  98056|        406|
|  98053|        405|
+-------+-----------+
only showing top 20 rows


In [9]:
#Visualización promedio
sql_str = "select zipcode, avg(price) as precio_promedio from housing group by zipcode"
spark.sql(sql_str).show()

+-------+------------------+
|zipcode|   precio_promedio|
+-------+------------------+
|  98002| 234284.0351758794|
|  98155|423725.69506726455|
|  98198| 302878.8821428571|
|  98146| 359483.2395833333|
|  98122| 634360.1793103449|
|  98077| 682774.8787878788|
|  98006| 859684.7791164658|
|  98001| 280804.6906077348|
|  98005|        810164.875|
|  98112| 1095499.342007435|
|  98115| 619900.5471698113|
|  98059|493552.53205128206|
|  98075| 790576.6545961003|
|  98023|286732.79158316634|
|  98109| 879623.6238532111|
|  98136| 551688.6730038023|
|  98052|  645231.456445993|
|  98011| 490351.4666666667|
|  98014| 455617.1129032258|
|  98058|353608.63516483514|
+-------+------------------+
only showing top 20 rows


In [10]:
#Visualización más completa
sql_str = "select zipcode, avg(price) as precio_promedio, avg(sqft_living) as tamano_promedio from housing group by zipcode order by precio_promedio desc"
spark.sql(sql_str).show()

+-------+------------------+------------------+
|zipcode|   precio_promedio|   tamano_promedio|
+-------+------------------+------------------+
|  98039|         2160606.6|            3800.9|
|  98004|1355927.0820189274|2909.0220820189274|
|  98040|1194230.0212765958|3106.8333333333335|
|  98112| 1095499.342007435|2498.7434944237916|
|  98102| 901258.2666666667|2159.7428571428572|
|  98109| 879623.6238532111|2054.7798165137615|
|  98105| 862825.2314410481|2150.5764192139736|
|  98006| 859684.7791164658|2888.2951807228915|
|  98119| 849448.0163043478|2005.6141304347825|
|  98005|        810164.875|2656.8035714285716|
|  98033| 803719.5231481482| 2381.340277777778|
|  98199|  791820.807570978|2161.7981072555203|
|  98075| 790576.6545961003|3016.3704735376045|
|  98074|  685605.775510204|2645.8707482993195|
|  98077| 682774.8787878788|2857.0454545454545|
|  98053| 678163.0592592593| 2620.879012345679|
|  98177| 676185.3921568628|2323.3333333333335|
|  98008| 645507.3780918728|2133.4452296

In [11]:
#Visualización vertical
spark.sql(sql_str).show(5,40,True)

-RECORD 0-----------------------------
 zipcode         | 98039              
 precio_promedio | 2160606.6          
 tamano_promedio | 3800.9             
-RECORD 1-----------------------------
 zipcode         | 98004              
 precio_promedio | 1355927.0820189274 
 tamano_promedio | 2909.0220820189274 
-RECORD 2-----------------------------
 zipcode         | 98040              
 precio_promedio | 1194230.0212765958 
 tamano_promedio | 3106.8333333333335 
-RECORD 3-----------------------------
 zipcode         | 98112              
 precio_promedio | 1095499.342007435  
 tamano_promedio | 2498.7434944237916 
-RECORD 4-----------------------------
 zipcode         | 98102              
 precio_promedio | 901258.2666666667  
 tamano_promedio | 2159.7428571428572 
only showing top 5 rows


In [12]:
#Plan de ejecución con explain
df.createOrReplaceTempView("housing")
sql_str = "select zipcode, avg(price) as precio_promedio, avg(sqft_living) as tamano_promedio from housing group by zipcode order by precio_promedio desc"
spark.sql(sql_str).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [precio_promedio#258 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(precio_promedio#258 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=270]
      +- HashAggregate(keys=[zipcode#32], functions=[avg(price#18), avg(sqft_living#21)])
         +- Exchange hashpartitioning(zipcode#32, 200), ENSURE_REQUIREMENTS, [plan_id=267]
            +- HashAggregate(keys=[zipcode#32], functions=[partial_avg(price#18), partial_avg(sqft_living#21)])
               +- FileScan csv [price#18,sqft_living#21,zipcode#32] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/Nefi/OneDrive/Documentos/09-EBAC/EBACMX-DATA-ANALYST/Refer..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<price:int,sqft_living:int,zipcode:int>




In [13]:
#Plan de ejecución con explain extendido
df.createOrReplaceTempView("housing")
sql_str = "select zipcode, avg(price) as precio_promedio, avg(sqft_living) as tamano_promedio from housing group by zipcode order by precio_promedio desc"
spark.sql(sql_str).explain(extended=True)

== Parsed Logical Plan ==
'Sort ['precio_promedio DESC NULLS LAST], true
+- 'Aggregate ['zipcode], ['zipcode, 'avg('price) AS precio_promedio#270, 'avg('sqft_living) AS tamano_promedio#271]
   +- 'UnresolvedRelation [housing], [], false

== Analyzed Logical Plan ==
zipcode: int, precio_promedio: double, tamano_promedio: double
Sort [precio_promedio#270 DESC NULLS LAST], true
+- Aggregate [zipcode#32], [zipcode#32, avg(price#18) AS precio_promedio#270, avg(sqft_living#21) AS tamano_promedio#271]
   +- SubqueryAlias housing
      +- View (`housing`, [id#17L, price#18, bedrooms#19, bathrooms#20, sqft_living#21, sqft_lot#22, floors#23, waterfront#24, view#25, condition#26, grade#27, sqft_above#28, sqft_basement#29, yr_built#30, yr_renovated#31, zipcode#32, lat#33, long#34, sqft_living15#35, sqft_lot15#36, date#37, sale_year#38, price_per_m2#39])
         +- Relation [id#17L,price#18,bedrooms#19,bathrooms#20,sqft_living#21,sqft_lot#22,floors#23,waterfront#24,view#25,condition#26,grade#27,sq

In [14]:
spark.sql(sql_str).columns

['zipcode', 'precio_promedio', 'tamano_promedio']

In [15]:
spark.sql(sql_str).count()

70

In [16]:
spark = SparkSession.builder.appName('Partitionby() PySpark').getOrCreate()

dataframe = spark.read.option("header", True).csv("/Users/Nefi/OneDrive/Documentos/09-EBAC/EBACMX-DATA-ANALYST/Referencias/kc_house_data_clean.csv")

dataframe.printSchema()

root
 |-- id: string (nullable = true)
 |-- price: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- sqft_living: string (nullable = true)
 |-- sqft_lot: string (nullable = true)
 |-- floors: string (nullable = true)
 |-- waterfront: string (nullable = true)
 |-- view: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sqft_above: string (nullable = true)
 |-- sqft_basement: string (nullable = true)
 |-- yr_built: string (nullable = true)
 |-- yr_renovated: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- sqft_living15: string (nullable = true)
 |-- sqft_lot15: string (nullable = true)
 |-- date: string (nullable = true)
 |-- sale_year: string (nullable = true)
 |-- price_per_m2: string (nullable = true)



In [19]:
dataframe.write.option("header", True)\
    .partitionBy("zipcode")\
    .mode("overwrite")\
    .csv("C:/temp/housing_partitioned/zipcode")

In [20]:
#Partición con año de venta
dataframe.write.option("header", True)\
    .partitionBy("sale_year")\
    .mode("overwrite")\
    .csv("C:/temp/housing_partitioned/sale_year")

In [21]:
#Partición con variables múltiples
dataframe.write.option("header", True)\
    .partitionBy("zipcode", "sale_year")\
    .mode("overwrite")\
    .csv("C:/temp/housing_partitioned/zipcode_sale_year")

In [22]:
df_test = spark.read.csv("C:/temp/housing_partitioned/zipcode_sale_year", header=True)
df_test.show()

+----------+-------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+--------+-------------+----------+----------+------------+-------+---------+
|        id|  price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|    lat|    long|sqft_living15|sqft_lot15|      date|price_per_m2|zipcode|sale_year|
+----------+-------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+--------+-------------+----------+----------+------------+-------+---------+
|6865200140| 485000|       4|        1|       1600|    4300|   1.5|         0|   0|        4|    7|      1600|            0|    1916|           0|47.6648|-122.343|         1610|      4300|2014-05-29|     3262.81|  98103|     2014|
|3362400431| 518500|       3|      3.5|       1590|    1102|     3|         

In [23]:
#Lectura
spark = SparkSession.builder.appName('coalesce() PySpark').getOrCreate()

dataco = spark.read.option("header", True).csv("/Users/Nefi/OneDrive/Documentos/09-EBAC/EBACMX-DATA-ANALYST/Referencias/kc_house_data_clean.csv")

dataco.printSchema()

root
 |-- id: string (nullable = true)
 |-- price: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- sqft_living: string (nullable = true)
 |-- sqft_lot: string (nullable = true)
 |-- floors: string (nullable = true)
 |-- waterfront: string (nullable = true)
 |-- view: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sqft_above: string (nullable = true)
 |-- sqft_basement: string (nullable = true)
 |-- yr_built: string (nullable = true)
 |-- yr_renovated: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- sqft_living15: string (nullable = true)
 |-- sqft_lot15: string (nullable = true)
 |-- date: string (nullable = true)
 |-- sale_year: string (nullable = true)
 |-- price_per_m2: string (nullable = true)



In [24]:
dataco.count()

21613

In [25]:
#Repartition 20
dataco.repartition(20)\
    .write.mode("overwrite")\
    .option("header", True)\
    .csv("C:/temp/partition/rep")

In [26]:
dataco.repartition(10)\
    .write.mode("overwrite")\
    .option("header", True)\
    .csv("C:/temp/partition/ej10")

In [27]:
#Coalesce
dataco_2 = dataco.repartition(20)
dataco_2.rdd.getNumPartitions()

20

In [28]:
dataco_3 = dataco_2.coalesce(10)
dataco_3.rdd.getNumPartitions()

10

In [29]:
dataco_3\
    .write.mode("overwrite")\
    .option("header", True)\
    .csv("C:/temp/partition/coalesce")

In [30]:
#PyArrow
from pyarrow import csv
import pyarrow as pa

In [31]:
archivo = '/Users/Nefi/OneDrive/Documentos/09-EBAC/EBACMX-DATA-ANALYST/Referencias/kc_house_data_clean.csv'
tab_housing = csv.read_csv(archivo)

In [32]:
tab_housing

pyarrow.Table
id: int64
price: int64
bedrooms: int64
bathrooms: double
sqft_living: int64
sqft_lot: int64
floors: double
waterfront: int64
view: int64
condition: int64
grade: int64
sqft_above: int64
sqft_basement: int64
yr_built: int64
yr_renovated: int64
zipcode: int64
lat: double
long: double
sqft_living15: int64
sqft_lot15: int64
date: date32[day]
sale_year: int64
price_per_m2: double
----
id: [[7129300520,6414100192,5631500400,2487200875,1954400510,...,3524039202,59000201,1782000085,3121500150,5207200360],[2570500230,2481600110,5104520150,7504110760,3649100276,...,8731950910,1726600110,3320000212,1670400068,7424700145],[2408800120,1776230180,7889601320,7589200153,8032700175,...,263000018,6600060120,1523300141,291310100,1523300157]]
price: [[221900,538000,180000,604000,510000,...,1065500,611206,350000,894000,420000],[400000,675000,426000,750000,368000,...,227000,675000,397500,206000,1190000],[360000,427500,115000,559000,420000,...,360000,400000,402101,400000,325000]]
bedrooms: [[3,3

In [33]:
len(tab_housing)

21613

In [34]:
tab_housing.column_names

['id',
 'price',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15',
 'date',
 'sale_year',
 'price_per_m2']

In [35]:
tab_housing.schema

id: int64
price: int64
bedrooms: int64
bathrooms: double
sqft_living: int64
sqft_lot: int64
floors: double
waterfront: int64
view: int64
condition: int64
grade: int64
sqft_above: int64
sqft_basement: int64
yr_built: int64
yr_renovated: int64
zipcode: int64
lat: double
long: double
sqft_living15: int64
sqft_lot15: int64
date: date32[day]
sale_year: int64
price_per_m2: double

In [36]:
tab_housing.num_columns

23

In [37]:
tab_housing.num_rows

21613

In [38]:
type(tab_housing)

pyarrow.lib.Table

In [47]:
#Agrupa la tabla por Género
import pyarrow.compute as pc

tab_zip = tab_housing.group_by("zipcode")\
    .aggregate([("price","sum")])

In [48]:
df1 = tab_zip.to_pandas()

In [49]:
df1.head()

,zipcode,price_sum
0,98010,42366599
1,98059,230982585
2,98146,103531173
3,98117,318967639
4,98103,352121365


In [50]:
tab2 = tab_housing.append_column("Test", pa.array(['0'] * len(tab_housing), pa.string()))
df2 = tab2.to_pandas()
df2.head()

,id,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,sale_year,price_per_m2,Test
0,7129300520,221900,3,1.00,1180,5650,1.0,0,0,3,...,0,98178,47.5112,-122.257,1340,5650,2014-10-13,2014,2024.16,0
1,6414100192,538000,3,2.25,2570,7242,2.0,0,0,3,...,1991,98125,47.7210,-122.319,1690,7639,2014-12-09,2014,2253.30,0
2,5631500400,180000,2,1.00,770,10000,1.0,0,0,3,...,0,98028,47.7379,-122.233,2720,8062,2015-02-25,2015,2516.24,0
3,2487200875,604000,4,3.00,1960,5000,1.0,0,0,5,...,0,98136,47.5208,-122.393,1360,5000,2014-12-09,2014,3317.04,0
4,1954400510,510000,3,2.00,1680,8080,1.0,0,0,3,...,0,98074,47.6168,-122.045,1800,7503,2015-02-18,2015,3267.61,0


In [51]:
#Uso de Parquet
import pyarrow.parquet as pq
import pandas as pd
import pyarrow as pa

df = pd.DataFrame({'lin1':[-20, 100, 200],
                   'lin2':['este', 'es un', 'ejemplo'],
                   'l3':[False, True, True]},
                   index = list('abc'))

In [52]:
df

,lin1,lin2,l3
a,-20,este,False
b,100,es un,True
c,200,ejemplo,True
